# Car Suspension Dynamics — Spring-Damper System Analysis
## AMME2500 Engineering Dynamics — Major Assignment

---

| | |
|:---|:---|
| **Name** | **SID** | **Contribution** |
| Isaac Edmonds | 550813846 | [one-line contribution statement] |
| [Name 2] | [xxxxxxxxx] | [one-line contribution statement] |
| [Name 3] | [xxxxxxxxx] | [one-line contribution statement] |

**Date Submitted:** 29th May 2026

## Abstract

*A brief summary of the method, results, and conclusions (typically 150–200 words).*

[Describe: (1) the system modelled and its governing physics, (2) the numerical method used to solve the equations of motion, (3) the key quantitative results obtained, and (4) the main engineering conclusions drawn from the analysis.]

## Declaration of Generative AI Use

- GitHub Copilot was used for autocomplete suggestions when writing numerical integration code.
- Claude was used for structuring jupyter notebook in a visually appealing way.

## 1. Introduction

*Outline the system being investigated, its novelty, the motivation, aims, and structure of the report.*

[Describe the car suspension spring-damper system — what it is, why it is an interesting and practically relevant mechanical system to study, what gaps or questions this investigation addresses, the specific aims of the report, and how the rest of the report is organised.]

This report is an investagation into car suspension systems. We are aim to determine the effectiveness of different suspension systems by finding a meaningful correlation between danmpening and spring cofficients, and the safety of the passeners. To do this we have made a computational model of our system. Our model represents the main body of the car as a beam suspended by two different spring and dampner systems on either side. There are also two passenegers in the system who are substituted for two pendulms with a torsion spring to simulate the human tendency to sit upright. This system is mechanically interesting as it combines the dampened spring and pendulum system...... [Finish later]

## 2. Methodology

### 2.1 System Description & Free Body Diagram

*Insert FBD sketch or image here.*

### 2.2 Equations of Motion

*Derive or present the governing ODEs here. Use LaTeX for equations, e.g.:*

$$m\ddot{x} + c\dot{x} + kx = F(t)$$

### 2.3 Computational Approach

*Describe the numerical integration method used (e.g. `solve_ivp` with RK45), step-size selection, solver tolerances, etc.*

### 2.4 Initial & Boundary Conditions

| Parameter | Symbol | Value | Units |
|:----------|:------:|------:|:-----:|
| [param]   | [sym]  | [val] | [unit]|

### 2.5 Validation & Convergence

*Demonstrate that the solver converges and produces physically consistent results (e.g. energy conservation checks, comparison against an analytical limit case, step-size sensitivity).*

---
### Code & Implementation

In [11]:
# Load the necessary libraries
import numpy as np
import sympy as sp
from sympy.physics.mechanics import dynamicsymbols
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# This is for pretty printing
import IPython.display as disp

print("hello world")

hello world


In [12]:
#global constants
g = 9.81

In [13]:
L, m_pl_a, m_pl_b, m_car, k_a, k_b, c_a, c_b, g = sp.symbols(
    'L m_pl_a m_pl_b m_car k_a k_b c_a c_b g'
)

t = sp.symbols('t')

x = sp.Function('x')(t)
a = sp.Function('a')(t)
b = sp.Function('b')(t)

theta_a = sp.Function('theta_a')(t)
theta_b = sp.Function('theta_b')(t)
theta_a_dot = sp.diff(theta_a, t)
theta_b_dot = sp.diff(theta_b, t)

F = sp.Function('F')(t)
F_equation = sp.Eq(F, sp.sin(2 * sp.pi * t / 5))

In [14]:
class Car:
    def __init__(self, L, m_ppl_a, m_ppl_b, m_car, k_a, k_b, c_a, c_b, theta_a_0, theta_b_0, theta_a_dot_0, theta_b_dot_0, x, a, b, t):
        # Initialize attributes
        self.L = L
        self.m_ppl_a = m_ppl_a
        self.m_ppl_b = m_ppl_b
        self.m_car = m_car
        self.k_a = k_a
        self.k_b = k_b
        self.c_a = c_a
        self.c_b = c_b
        self.theta_a_0 = theta_a_0
        self.theta_b_0 = theta_b_0
        self.theta_a_dot_0 = theta_a_dot_0
        self.theta_b_dot_0 = theta_b_dot_0
        self.m_eff_a = m_ppl_a * L / 2
        self.m_eff_b = m_ppl_b * L / 2
        self.a = a
        self.b = b
        self.x = x
        self.t = t

    #naimish's equation for theta 
    def theta_equation(self):
        return (self.b-self.a)/self.L
    
    #naimsh's equation for x
    def x_equation(self):
        return (self.a + self.b)

    #naimish's equation for sprint and dampening forces on the car
    def a_b_offset_equation(self, a_or_b): # enter 'a' or 'b' in a_or_b
        var = self.a if a_or_b == 'a' else self.b
        c_val = self.c_a if a_or_b == 'a' else self.c_b
        m_eff_val = self.m_eff_a if a_or_b == 'a' else self.m_eff_b
        k_val = self.k_a if a_or_b == 'a' else self.k_b
        return (m_eff_val) * sp.diff(var, self.t, 2) + (c_val) * sp.diff(var, self.t) + (k_val) * var + F_equation.rhs
    
    def solve_for_a_b(self):
        a_b_eq = self.a_b_offset_equation('a'), self.a_b_offset_equation('b')
        a_b_sol = sp.solve(a_b_eq, (sp.diff(self.a, self.t, 2), sp.diff(self.b, self.t, 2)))
        return a_b_sol

    def equations_of_motion(self):
        theta_eq = self.theta_equation()
        x_eq = self.x_equation()
        a_offset_eq = self.a_b_offset_equation('a')
        b_offset_eq = self.a_b_offset_equation('b')
        return [theta_eq, x_eq, a_offset_eq, b_offset_eq]
    
# def plot_solution(sol, figsize=(12, 5)):
#     fig, ax = plt.subplots(1, 3, figsize=figsize)
#     ax[0].plot(sol.t, sol.y[0], label='x', linewidth=2)
#     ax[0].plot(sol.t, sol.y[1], label='y', linewidth=2)
#     ax[0].set_ylabel('Displacement', fontsize=14)
#     ax[0].legend()
#     ax[0].grid(which='both')
#     ax[0].set_xlim(t_span)
#     ax[1].plot(sol.y[0], sol.y[1])
#     ax[1].set_ylabel('y', fontsize=14)
#     ax[1].set_xlabel('x', fontsize=14)
#     ax[1].grid(which='both')
#     ax[1].axis('equal')
#     ax[1].set_title('Trajectory of the bead')
#     ax[2].plot(sol.y[2], sol.y[3], linewidth=2)
#     ax[2].set_ylabel('Vx', fontsize=14)
#     ax[2].set_xlabel('Vy', fontsize=14)
#     ax[2].grid(which='both')
#     ax[2].axis('equal')
#     fig.tight_layout(pad=0.3)
#     ax[2].set_title('Velocity of the bead')
#     return fig, ax


In [18]:
car1 = Car(L, m_pl_a, m_pl_b, m_car, k_a, k_b, c_a, c_b, theta_a, theta_b, theta_a_dot, theta_b_dot, x, a, b, t)

print(car1.__dict__)

print(car1.solve_for_a_b())

{'L': L, 'm_ppl_a': m_pl_a, 'm_ppl_b': m_pl_b, 'm_car': m_car, 'k_a': k_a, 'k_b': k_b, 'c_a': c_a, 'c_b': c_b, 'theta_a_0': theta_a(t), 'theta_b_0': theta_b(t), 'theta_a_dot_0': Derivative(theta_a(t), t), 'theta_b_dot_0': Derivative(theta_b(t), t), 'm_eff_a': L*m_pl_a/2, 'm_eff_b': L*m_pl_b/2, 'a': a(t), 'b': b(t), 'x': x(t), 't': t}
{Derivative(a(t), (t, 2)): -2*c_a*Derivative(a(t), t)/(L*m_pl_a) - 2*k_a*a(t)/(L*m_pl_a) - 2*sin(2*pi*t/5)/(L*m_pl_a), Derivative(b(t), (t, 2)): -2*c_b*Derivative(b(t), t)/(L*m_pl_b) - 2*k_b*b(t)/(L*m_pl_b) - 2*sin(2*pi*t/5)/(L*m_pl_b)}


## 3. Results

*Present sufficient figures, tables, calculations, and discussion to analyse the system and reach conclusions.*

**Reminder:** all graphs must include axis labels with units, a legend, and a descriptive caption.

---

## 4. Conclusions

*Summary of what was done and the main scientific/engineering findings.*

[Concisely restate the aims, the approach taken, and the key quantitative/qualitative findings. Avoid introducing new material here.]

---
## Appendix

*Additional material useful but not essential to the main narrative. This section is optional.*

---
## References

*All references must be properly formatted with corresponding in-text citations.*

[1] Author(s), "Title," *Journal/Book*, vol. X, no. Y, pp. Z–Z, Year.

[2] ...